# Resume an incomplete SIRIUS run

For C24 alkamine trihydroxy part 4 the SIRIUS run stopped early. This removes the spectra that already have a SIRIUS result from the input MGF so SIRIUS can be run on the remaining ones.

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(f"{PROJECT_DIR}/SIRIUS_outputs/C24_alkamine_trihydroxy_part4_incomplete_summaries/formula_identifications.tsv", sep='\t')
df['mappingFeatureId'] = df['mappingFeatureId'].str.split("UNKNOWN_").str[1]
calculated = df['mappingFeatureId'].to_list()

In [ ]:
from pathlib import Path

mgf_path = Path(f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2_scan.mgf")
out_path = mgf_path.with_name(f"{mgf_path.stem}_not_calculated_filtered.mgf")

# Make matching robust (handles numeric IDs and surrounding spaces).
calculated_set = {str(x).strip() for x in calculated if pd.notna(x)}

with mgf_path.open("r", encoding="utf-8") as f:
    lines = f.readlines()

filtered_lines = []
current_block = []
in_block = False
current_title = None
removed = 0
kept = 0

for line in lines:
    stripped = line.strip()

    if stripped == "BEGIN IONS":
        in_block = True
        current_block = [line]
        current_title = None
        continue

    if in_block:
        current_block.append(line)
        if stripped.startswith("TITLE="):
            current_title = stripped.split("=", 1)[1].strip()

        if stripped == "END IONS":
            title_match = current_title in calculated_set if current_title is not None else False
            if not title_match:
                filtered_lines.extend(current_block)
                kept += 1
            else:
                removed += 1

            in_block = False
            current_block = []
            current_title = None
        continue

    # Keep any content outside ion blocks as-is.
    filtered_lines.append(line)

with out_path.open("w", encoding="utf-8") as f:
    f.writelines(filtered_lines)

print(f"Input scans kept: {kept}")
print(f"Input scans removed (TITLE in calculated): {removed}")
print(f"Filtered MGF written to: {out_path}")